In [1]:
import pandas as pd

# 전처리 데이터 불러오기
df = pd.read_csv("data/processed/electricity_clean.csv")

print(df.shape)
print(df.head())

(110040, 7)
     연도     시도  시군구 계약종별  월        사용량       날짜
0  2021  서울특별시  종로구  주택용  1   25471662  2021-01
1  2021  서울특별시  종로구  일반용  1  113082168  2021-01
2  2021  서울특별시  종로구  교육용  1   10072542  2021-01
3  2021  서울특별시  종로구  산업용  1    6243796  2021-01
4  2021  서울특별시  종로구  농사용  1       5390  2021-01


In [2]:
# 군위군 데이터 확인
gunwi = df[df["시군구"] == "군위군"]

print(gunwi.groupby(["연도", "시도"]).size())

연도    시도   
2021  경상북도     96
2022  경상북도     96
2023  경상북도     96
      대구광역시    96
2024  대구광역시    96
2025  대구광역시    96
dtype: int64


In [3]:
# 개성시 데이터 확인
gaeseong = df[df["시군구"] == "개성시"]

print(gaeseong.groupby("연도")["사용량"].agg(
    ["count", "sum", "min", "max"]
))

      count  sum  min  max
연도                        
2021     48    0    0    0
2022     48    0    0    0


In [4]:
# 전처리
# 원본 데이터 보존
df_model = df.copy()

# 1. 개성시 제외
df_model = df_model[
    df_model["시군구"] != "개성시"
].copy()

# 2. 군위군 지역명 통일
df_model.loc[
    df_model["시군구"] == "군위군",
    "시도"
] = "대구광역시"

print("전처리 후 데이터 크기:", df_model.shape)

print(
    df_model[df_model["시군구"] == "군위군"]
    .groupby(["연도", "시도"])
    .size()
)

전처리 후 데이터 크기: (109944, 7)
연도    시도   
2021  대구광역시     96
2022  대구광역시     96
2023  대구광역시    192
2024  대구광역시     96
2025  대구광역시     96
dtype: int64


In [5]:
# 지역명 통일 후 중복 확인
keys = ["시도", "시군구", "계약종별", "날짜"]

duplicates = df_model[
    df_model.duplicated(subset=keys, keep=False)
]

print("중복 행 개수:", len(duplicates))

print(
    duplicates.groupby(["연도", "시군구"]).size()
)

중복 행 개수: 192
연도    시군구
2023  군위군    192
dtype: int64


In [6]:
#2023년 중복 데이터 통합
# 지역과 날짜, 계약종별 기준으로 사용량 합산
keys = ["연도", "시도", "시군구", "계약종별", "월", "날짜"]

df_model = (
    df_model
    .groupby(keys, as_index=False)["사용량"]
    .sum()
)

print("통합 후 데이터 크기:", df_model.shape)

통합 후 데이터 크기: (109848, 7)


In [7]:
# 지역과 날짜, 계약종별 기준으로 사용량 합산
keys = ["연도", "시도", "시군구", "계약종별", "월", "날짜"]

df_model = (
    df_model
    .groupby(keys, as_index=False)["사용량"]
    .sum()
)

print("통합 후 데이터 크기:", df_model.shape)

통합 후 데이터 크기: (109848, 7)


In [8]:
# 통합 결과 검증
# 중복 확인
keys = ["시도", "시군구", "계약종별", "날짜"]

print("중복 개수:", df_model.duplicated(subset=keys).sum())

# 군위군 연도별 데이터 개수
print(
    df_model[df_model["시군구"] == "군위군"]
    .groupby("연도")
    .size()
)

# 음수 확인
print("음수 개수:", (df_model["사용량"] < 0).sum())

중복 개수: 0
연도
2021    96
2022    96
2023    96
2024    96
2025    96
dtype: int64
음수 개수: 0


In [9]:
# 합계 데이터 제외
# 합계 제외하고 세부 계약종별만 추출
df_train = df_model[
    df_model["계약종별"] != "합 계"
].copy()

print("학습 데이터 크기:", df_train.shape)
print(df_train["계약종별"].value_counts())

학습 데이터 크기: (96108, 7)
계약종별
가로등    13740
교육용    13740
산업용    13740
일반용    13740
심 야    13740
주택용    13740
농사용    13668
Name: count, dtype: int64


In [10]:
# 날짜순으로 정렬
# 날짜 형식 변환
df_train["날짜"] = pd.to_datetime(df_train["날짜"])

# 지역, 계약종별, 날짜순 정렬
df_train = df_train.sort_values(
    ["시도", "시군구", "계약종별", "날짜"]
).reset_index(drop=True)

print(df_train.head(10))

     연도   시도  시군구 계약종별   월         날짜      사용량
0  2021  강원도  강릉시  가로등   1 2021-01-01  2474109
1  2021  강원도  강릉시  가로등   2 2021-02-01  2320006
2  2021  강원도  강릉시  가로등   3 2021-03-01  2177804
3  2021  강원도  강릉시  가로등   4 2021-04-01  2194475
4  2021  강원도  강릉시  가로등   5 2021-05-01  2116684
5  2021  강원도  강릉시  가로등   6 2021-06-01  2066449
6  2021  강원도  강릉시  가로등   7 2021-07-01  2038350
7  2021  강원도  강릉시  가로등   8 2021-08-01  2195802
8  2021  강원도  강릉시  가로등   9 2021-09-01  2170576
9  2021  강원도  강릉시  가로등  10 2021-10-01  2312138


In [11]:
# 과거 사용량 생성
# 지역과 계약종별 기준으로 묶기
group_cols = ["시도", "시군구", "계약종별"]

grouped = df_train.groupby(group_cols)["사용량"]

# 이전 달 사용량
df_train["lag_1"] = grouped.shift(1)

# 전년도 같은 달 사용량
df_train["lag_12"] = grouped.shift(12)

print(df_train.head(15))

      연도   시도  시군구 계약종별   월         날짜      사용량      lag_1     lag_12
0   2021  강원도  강릉시  가로등   1 2021-01-01  2474109        NaN        NaN
1   2021  강원도  강릉시  가로등   2 2021-02-01  2320006  2474109.0        NaN
2   2021  강원도  강릉시  가로등   3 2021-03-01  2177804  2320006.0        NaN
3   2021  강원도  강릉시  가로등   4 2021-04-01  2194475  2177804.0        NaN
4   2021  강원도  강릉시  가로등   5 2021-05-01  2116684  2194475.0        NaN
5   2021  강원도  강릉시  가로등   6 2021-06-01  2066449  2116684.0        NaN
6   2021  강원도  강릉시  가로등   7 2021-07-01  2038350  2066449.0        NaN
7   2021  강원도  강릉시  가로등   8 2021-08-01  2195802  2038350.0        NaN
8   2021  강원도  강릉시  가로등   9 2021-09-01  2170576  2195802.0        NaN
9   2021  강원도  강릉시  가로등  10 2021-10-01  2312138  2170576.0        NaN
10  2021  강원도  강릉시  가로등  11 2021-11-01  2414784  2312138.0        NaN
11  2021  강원도  강릉시  가로등  12 2021-12-01  2465418  2414784.0        NaN
12  2022  강원도  강릉시  가로등   1 2022-01-01  2473322  2465418.0  2474109.0
13  2022  강원도  강릉시  

In [12]:
# 실제로 월이 연속되는지 검증
# 같은 지역·계약종별의 이전 기록 날짜
df_train["이전날짜"] = (
    df_train.groupby(group_cols)["날짜"].shift(1)
)

# 이전 기록과 현재 기록의 월 차이
month_number = df_train["연도"] * 12 + df_train["월"]

previous_month_number = (
    month_number.groupby(
        [df_train[col] for col in group_cols]
    ).shift(1)
)

month_gap = month_number - previous_month_number

# 이전 기록이 있지만 1개월 차이가 아닌 경우
gaps = df_train[
    df_train["이전날짜"].notna() &
    (month_gap != 1)
]

print("월 연속성이 깨진 행:", len(gaps))
print(gaps[["시도", "시군구", "계약종별", "날짜", "이전날짜"]].head(10))

월 연속성이 깨진 행: 0
Empty DataFrame
Columns: [시도, 시군구, 계약종별, 날짜, 이전날짜]
Index: []


In [13]:
# 과거 사용량이 없는 행 제거
# lag 변수의 결측치 확인
print(df_train[["lag_1", "lag_12"]].isna().sum())

# 두 변수가 모두 존재하는 데이터만 사용
df_ready = df_train.dropna(
    subset=["lag_1", "lag_12"]
).copy()

print("정리 후 데이터 크기:", df_ready.shape)
print(df_ready["연도"].value_counts().sort_index())

lag_1      1603
lag_12    19236
dtype: int64
정리 후 데이터 크기: (76872, 10)
연도
2022    19224
2023    19212
2024    19224
2025    19212
Name: count, dtype: int64


In [14]:
# 학습 검증 테스터 분리
# 연도별 데이터 분리
train = df_ready[df_ready["연도"] <= 2023].copy()
valid = df_ready[df_ready["연도"] == 2024].copy()
test = df_ready[df_ready["연도"] == 2025].copy()

print("학습 데이터:", train.shape)
print("검증 데이터:", valid.shape)
print("테스트 데이터:", test.shape)

학습 데이터: (38436, 10)
검증 데이터: (19224, 10)
테스트 데이터: (19212, 10)


In [15]:
# 학습, 검증, 테스트 데이터 저장
train.to_csv(
    "data/processed/train.csv",
    index=False,
    encoding="utf-8-sig"
)

valid.to_csv(
    "data/processed/valid.csv",
    index=False,
    encoding="utf-8-sig"
)

test.to_csv(
    "data/processed/test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("데이터 저장 완료!")

데이터 저장 완료!
